# Lesson 11: GPU, secrets, security (solution)

**Goal:** configure the app from the environment and make sure secrets never show up in logs.

**Before you start**
- Theory: [lesson 11](../../../docs/lessons/11_gpu_secrets_security.md)
- Python concepts: `os.environ`, pydantic-settings, `SecretStr`
- Files used: `src/docrag/shared/settings.py`. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import os

import onnxruntime

from docrag.shared.settings import Settings

# Remember the real values so this notebook leaves your environment unchanged.
_saved = {k: os.environ.get(k) for k in ("DOCRAG_BUDGET_EUR", "OPENAI_API_KEY")}


def restore_env() -> None:
    for k, v in _saved.items():
        if v is None:
            os.environ.pop(k, None)
        else:
            os.environ[k] = v

## Step 1: Settings come from the environment

`Settings` fields are filled from environment variables with the prefix `DOCRAG_` (and from a local `.env`). Values are **validated and converted**: `"1.5"` becomes a float, `"abc"` would fail at startup instead of deep inside a run. `use_keyring=False` skips the OS credential store.

In [ ]:
os.environ["DOCRAG_BUDGET_EUR"] = "1.5"
settings = Settings(use_keyring=False)

In [ ]:
try:
    assert settings is not None, (
        "Hint: settings = Settings(use_keyring=False) after setting the variable"
    )
    assert settings.budget_eur == 1.5 and isinstance(settings.budget_eur, float), (
        "The string became a float"
    )
    print("budget:", settings.budget_eur)
    print("OK")
finally:
    restore_env()

## Step 2: Secrets stay hidden

API keys are `SecretStr`: `repr()` and `str()` print `**********`, so `print(settings)` or a log line can't leak them. You must call `.get_secret_value()` explicitly where the key is really needed. `OPENAI_API_KEY` is read without the prefix, because every tool uses that name.

In [ ]:
os.environ["OPENAI_API_KEY"] = "sk-test-not-a-real-key"
secret_settings = Settings(use_keyring=False)

In [ ]:
try:
    assert secret_settings is not None, (
        "Hint: secret_settings = Settings(use_keyring=False)"
    )
    key = secret_settings.openai_api_key
    assert key is not None and key.get_secret_value() == "sk-test-not-a-real-key"
    assert "sk-test" not in repr(secret_settings) and "sk-test" not in str(key), (
        "SecretStr must hide the value"
    )
    print(repr(key))
    print("OK")
finally:
    restore_env()

## GPU check (just run it)

ONNX Runtime picks the first available execution provider. On a machine without NVIDIA GPU (CI, Docker) it's `CPUExecutionProvider`, and the same model files still work.

In [ ]:
print(onnxruntime.get_available_providers())

## Observation question

`SecretStr` hides the key in `repr`. Where could it still leak?

<details>
<summary>Answer</summary>

Anywhere code calls `.get_secret_value()` and passes the result on: an exception that includes request headers, a debug log of the HTTP request, a tracing attribute, an f-string in an error message. Outside the code: a committed `.env` (it's in `.gitignore`, as are `.env.*` files), shell history from `export KEY=...`, a Docker image layer that copied `.env`, or a screenshot. The project stores keys in the OS keyring (`keyring set docrag OPENROUTER_API_KEY`) so they never touch a file in the repo.
</details>

## The real code

- Settings: [`src/docrag/shared/settings.py`](../../../src/docrag/shared/settings.py)
- ONNX providers: [`src/docrag/embed/onnx.py`](../../../src/docrag/embed/onnx.py)
- Lesson: [`docs/lessons/11_gpu_secrets_security.md`](../../../docs/lessons/11_gpu_secrets_security.md)